<a href="https://colab.research.google.com/github/adirav96/Cloud/blob/main/TIGERHW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 34.3 MB/s eta 0:00:00


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import random
import time

In [ ]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY='' # <<< REPLACE THIS WITH YOUR ACTUAL API KEY
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Initialize the Gemini API
llm_model = genai.GenerativeModel('models/gemini-3.5-flash') # Using the model chosen by the user

In [ ]:
import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re

nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)

documents = {
    1: "Deceptive pollination in orchids a review of its patterns mechanisms and evolution",
    2: "The medicinal orchid as a source of potential bioactive compounds a review",
    3: "Orchid mycorrhizal fungi structure function and diversity",
    4: "Molecular phylogenetics of Orchidaceae a review",
    5: "An overview of orchid conservation challenges and climate directions"
}

target_terms = [
    'orchid', 'pollination', 'deceptive', 'mechanism', 'evolution',
    'medicinal', 'bioactive', 'compound', 'mycorrhizal', 'fungi',
    'structure', 'diversity', 'molecular', 'phylogenetics', 'conservation',
    'climate', 'pattern', 'potential', 'function', 'challenge'
]

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

inverted_index = {term: set() for term in target_terms}

for doc_id, text in documents.items():
    words = re.findall(r'\b\w+\b', text.lower())
    for word in words:
        if word not in stop_words:
            lemma = lemmatizer.lemmatize(word)
            if lemma in inverted_index:
                inverted_index[lemma].add(doc_id)

index_data = [{'term': term, 'DocIDs': list(doc_ids)} for term, doc_ids in inverted_index.items()]
df_index = pd.DataFrame(index_data)

print("--- Inverted Index ---")
print(df_index.to_string(index=False))

--- Inverted Index ---
         term       DocIDs
       orchid [1, 2, 3, 5]
  pollination          [1]
    deceptive          [1]
    mechanism          [1]
    evolution          [1]
    medicinal          [2]
    bioactive          [2]
     compound          [2]
  mycorrhizal          [3]
        fungi          [3]
    structure          [3]
    diversity          [3]
    molecular          [4]
phylogenetics          [4]
 conservation          [5]
      climate          [5]
      pattern          [1]
    potential          [2]
     function          [3]
    challenge          [5]


In [ ]:
# Install the Firebase Admin SDK
!pip install firebase-admin -q

In [ ]:
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore
import os

# Path to your service account key file
# Make sure you upload this file to your Colab environment
SERVICE_ACCOUNT_KEY_PATH = '/content/sample_data/cloudhw2-ed71f-firebase-adminsdk-fbsvc-5655fbbe87.json' # Updated path

# Initialize Firebase Admin SDK only if it hasn't been initialized yet
if not firebase_admin._apps:
    if os.path.exists(SERVICE_ACCOUNT_KEY_PATH):
        cred = credentials.Certificate(SERVICE_ACCOUNT_KEY_PATH)
        firebase_admin.initialize_app(cred)
        print("✅ Firebase Admin SDK initialized successfully!")
    else:
        print(f"❌ Error: Service account key file not found at {SERVICE_ACCOUNT_KEY_PATH}.")
        print("Please upload your 'serviceAccountKey.json' file or update the path.")
else:
    print("Firebase Admin SDK already initialized.")

# Get a Firestore client
db = firestore.client()

✅ Firebase Admin SDK initialized successfully!


In [ ]:
# Save the inverted_index to Firestore

# Convert sets in inverted_index to lists for JSON serialization
firebase_inverted_index = {term: list(doc_ids) for term, doc_ids in inverted_index.items()}

# Store the index in a document in a collection named 'inverted_indexes'
# You can choose a specific document ID, e.g., 'orchid_index'

try:
    doc_ref = db.collection('inverted_indexes').document('orchid_index')
    doc_ref.set(firebase_inverted_index)
    print("✅ Inverted index successfully saved to Firestore!")
    print("You can view it in your Firebase console under Firestore Database.")
except Exception as e:
    print(f"❌ Error saving inverted index to Firestore: {e}")

✅ Inverted index successfully saved to Firestore!
You can view it in your Firebase console under Firestore Database.


In [ ]:
# Updated RAG Pipeline using the EcologicalRAG instance
def rag_pipeline_updated(query):
    if 'rag_system' not in globals():
        return "❌ System not found. Please run the Initialization cell (Cell 8) first."

    try:
        # Query the live system
        result = rag_system.query(query)
        context = result.get('context', '')

        if not context:
            return "🔍 No relevant information found in your PDFs for this query."

        print("--- Retrieved Context from your PDFs ---")
        for meta in result.get('metadata', []):
            print(f"Source: {meta.get('title', 'Unknown Paper')}")

        return f"Retrieved context length: {len(context)} characters. Ready for LLM generation."
    except Exception as e:
        return f"❌ Error: {str(e)}. Please try re-running the initialization cell."

user_query = "What is the genomic diversity of orchids?"
print(rag_pipeline_updated(user_query))

❌ System not found. Please run the Initialization cell (Cell 8) first.


In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

class SimpleVectorStore:
    def __init__(self):
        self.embeddings = None
        self.documents = []
        self.metadatas = []

    def add(self, embeddings, documents, metadatas, ids=None):
        self.embeddings = np.array(embeddings) if self.embeddings is None else np.vstack([self.embeddings, embeddings])
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)

    def query(self, query_embedding, n_results=3):
        if self.embeddings is None or len(self.documents) == 0:
            return {"documents": [[]], "metadatas": [[]]}
        sims = np.dot(self.embeddings, query_embedding[0]) / (np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-8)
        idx = np.argsort(sims)[-n_results:][::-1]
        return {"documents": [[self.documents[i] for i in idx]], "metadatas": [[self.metadatas[i] for i in idx]]}

class EcologicalRAG:
    def __init__(self, **kwargs):
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
        self.collection = SimpleVectorStore()
        self.fitted = False
        self.use_chromadb = False
        print("🌊 Ecological RAG System Initialized")

    def preprocess_text(self, text):
        return re.sub(r'\s+', ' ', text).strip() if text else ""

    def extract_entities(self, text):
        return {'species': [], 'locations': [], 'methods': []}

    def generate_embeddings(self, texts):
        if not self.fitted:
            self.tfidf.fit(texts)
            self.fitted = True
        return self.tfidf.transform(texts).toarray()

    def load_papers(self, papers_data):
        print(f"📚 Processing {len(papers_data)} papers...")
        docs, metas = [], []
        for p in papers_data:
            abstract = p.get('abstract', '')
            title = p.get('title', '')
            full_text = f"{title} {abstract}"
            clean_text = self.preprocess_text(full_text)

            if len(clean_text) > 50:
                docs.append(clean_text)
                metas.append(p)

        if docs:
            embeddings = self.generate_embeddings(docs)
            self.collection.add(embeddings, docs, metas)
            print(f"✅ Successfully loaded {len(docs)} documents into the system.")

    def query(self, query_text):
        query_vector = self.generate_embeddings([query_text])
        results = self.collection.query(query_vector)
        context = "\n---\n".join(results['documents'][0])
        return {"context": context, "metadata": results['metadatas'][0], "papers_found": len(results['documents'][0])}

In [ ]:
# CELL 5: Data Loading Methods
# ============================
"""
📚 CELL 5: DATA LOADING METHODS
This cell adds data loading capabilities to the RAG system.
"""

def add_load_papers_method():
    """Add load_papers method to EcologicalRAG class"""

    def load_papers(self, papers_data):
        """Load papers into the RAG system"""
        print(f"📚 Loading {len(papers_data)} papers...")

        valid_papers = [p for p in papers_data if p.get('abstract', '').strip()]
        print(f"📖 Found {len(valid_papers)} papers with abstracts")

        if not valid_papers:
            print("❌ No valid papers found!")
            return

        documents, metadatas, ids = [], [], []

        for i, paper in enumerate(valid_papers):
            # Combine title and abstract
            text = f"{paper.get('title', '')} {paper.get('abstract', '')}"
            text = self.preprocess_text(text)

            if len(text) < 50:
                continue

            entities = self.extract_entities(text)

            metadata = {
                'title': paper.get('title', 'Unknown'),
                'authors': paper.get('authors', 'Unknown'),
                'journal': paper.get('journal', 'Unknown'),
                'year': paper.get('year', 2022),
                'doi': paper.get('doi', ''),
                'species': ', '.join(entities['species']),
                'locations': ', '.join(entities['locations']),
                'methods': ', '.join(entities['methods'])
            }

            documents.append(text)
            metadatas.append(metadata)
            ids.append(f"paper_{i}")

        if not documents:
            print("❌ No processable documents found!")
            return

        # Generate embeddings
        print("🔄 Generating embeddings...")
        embeddings = self.generate_embeddings(documents)

        # Add to vector store
        print("💾 Adding to vector store...")
        if self.use_chromadb:
            self.collection.add(
                embeddings=embeddings.tolist(),
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
        else:
            self.collection.add(
                embeddings=embeddings,
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )

        self.papers = valid_papers
        print(f"✅ Successfully loaded {len(documents)} papers!")

    # Add method to class
    EcologicalRAG.load_papers = load_papers

# Apply the method
add_load_papers_method()

print("✅ Data loading methods added!")
print("📋 Next: Run Cell 6 for search and query methods")

✅ Data loading methods added!
📋 Next: Run Cell 6 for search and query methods


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Image
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import re

# --- Screen 1: Image Upload ---
upload_label = widgets.HTML(value="<h3>Screen 1: Upload Orchid Image for Diagnosis</h3>")
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Choose Image'
)
image_preview = widgets.Image(format='png', width=300, height=300, layout=widgets.Layout(display='none'))
diagnosis_output = widgets.HTML(value='<p>...</p>')
upload_output = widgets.Output()

def on_upload_change(change):
    with upload_output:
        clear_output()
        if uploader.value:
            uploaded_file = uploader.value[list(uploader.value.keys())[0]]
            image_data = uploaded_file['content']
            image_preview.value = image_data
            image_preview.layout.display = 'block' # Show image preview
            print("Image uploaded successfully! (Stored in system memory)")
            print("Analyzing image for diagnosis...")
            time.sleep(1) # Simulate analysis time

            # Simulate diagnosis
            diagnosis = random.choice([
                "The plant looks healthy!",
                "Minor issue detected: slight spots on leaves. Consider checking humidity.",
                "Problem detected: brown spots on leaves. May need more watering or fertilization."
            ])
            diagnosis_output.value = f'<p><b>Diagnosis Result:</b> {diagnosis}</p>'
            print("Note: In a full system, this image could be linked to other screens for context or historical tracking.")

uploader.observe(on_upload_change, names='value')
image_upload_screen = widgets.VBox([
    upload_label, uploader, image_preview, diagnosis_output, upload_output
], layout=widgets.Layout(border='2px solid #ADD8E6', padding='10px', margin='5px', background_color='#F0F8FF'))

# --- Screen 2: IoT Sensor Data ---
iot_label = widgets.HTML(value="<h3>Screen 2: Sample Data from Orchid Sensors (IoT)</h3>")

temp_slider = widgets.FloatSlider(value=20.0, min=10.0, max=40.0, step=0.1, description='Temperature (C° - Current Reading):', disabled=True)
humidity_slider = widgets.FloatSlider(value=50.0, min=0.0, max=100.0, step=1.0, description='Humidity (% - Current Reading):', disabled=True)
soil_slider = widgets.FloatSlider(value=30.0, min=0.0, max=100.0, step=1.0, description='Soil Moisture (% - Current Reading):', disabled=True)

temp_status = widgets.HTML()
humidity_status = widgets.HTML()
soil_status = widgets.HTML() # Corrected typo from soli_status

sample_button = widgets.Button(description="Get Data Now", button_style='info')
iot_output = widgets.Output()

# sms_output widget is removed as it will be replaced by pop-ups

# Define optimal ranges for orchids (example values)
OPTIMAL_TEMP = (20, 28)
OPTIMAL_HUMIDITY = (50, 70)
OPTIMAL_SOIL = (40, 60)

def generate_smart_alerts(temp, humidity, soil):

    alerts = []

    if temp > 30:
        alerts.append("Temperature is too high")

    elif temp < 18:
        alerts.append("Temperature is too low")

    if humidity < 45:
        alerts.append("Air humidity is too low")

    if soil < 30:
        alerts.append("Soil moisture is too low")

    elif soil > 80:
        alerts.append("Soil moisture is too high")

    return alerts

# This function will now display a popup instead of returning HTML
def display_popup_alert(alerts_list):
    current_time = time.strftime("%H:%M")
    if alerts_list:
        alert_message = f"<h4>📱 Plant Alert! ({current_time})</h4>" \
                        f"<p><b>Issue(s):</b><br>- {'<br>- '.join(alerts_list)}</p>"
        alert_style = "background:#fff5f5; border:2px solid red;"
    else:
        alert_message = f"<h4>✅ Plant Status: All Optimal ({current_time})</h4>"
        alert_style = "background:#f5fff5; border:2px solid green;"

    popup_html = widgets.HTML(
        value=f"""
        <div style='padding:10px; border-radius:10px; {alert_style}'>
            {alert_message}
        </div>
        """
    )
    # Using an Output widget to display the popup temporarily
    popup_output = widgets.Output(layout={'border': '1px solid black', 'margin': '10px 0', 'padding': '5px'})
    with popup_output:
        display(popup_html)
    display(popup_output)

def get_status_html(value, optimal_range, label):
    status = ""
    color = "black"
    if value < optimal_range[0]:
        status = "Too Low ⬇️"
        color = "red"
    elif value > optimal_range[1]:
        status = "Too High ⬆️"
        color = "red"
    else:
        status = "Optimal ✅"
        color = "green"
    return f'<p style="color:{color};"><b>{label}:</b> {value} - {status}</p>'

# Removed extract_text_from_html as it's no longer needed for popups

def sample_iot_data(b):
    with iot_output:
        clear_output()
        print("Connecting to sensors... Sampling data... (Loading...)") # Added loading feedback

        temp_val = 32
        humidity_val = 40
        soil_val = 25

        temp_slider.value = temp_val
        humidity_slider.value = humidity_val
        soil_slider.value = soil_val

        temp_status.value = get_status_html(temp_val, OPTIMAL_TEMP, "Temperature")
        humidity_status.value = get_status_html(humidity_val, OPTIMAL_HUMIDITY, "Humidity")
        soil_status.value = get_status_html(soil_val, OPTIMAL_SOIL, "Soil Moisture")

        alerts = generate_smart_alerts(temp_val, humidity_val, soil_val)

        display_popup_alert(alerts) # Display the popup directly

        print("Data updated successfully.")

    # Removed the explicit print of SMS content to standard output

sample_button.on_click(sample_iot_data)

iot_sensor_screen = widgets.VBox([
    iot_label,
    sample_button,
    temp_slider,
    temp_status,
    humidity_slider,
    humidity_status,
    soil_slider,
    soil_status,
    iot_output
], layout=widgets.Layout(border='2px solid #D8BFD8', padding='10px', margin='5px', background_color='#F8F0FF')) # Added styling

# --- Screen 3: Search Engine ---
search_label = widgets.HTML(value="<h3>Screen 3: Ask About My Plant</h3>")

search_input = widgets.Textarea(
    value='',
    placeholder="Type your question here... (e.g., 'How to care for my orchid?')",
    description='Query:',
    layout=widgets.Layout(width='80%', height='60px')
)

search_button = widgets.Button(description="Search Info", button_style='primary', icon='search')
clear_button = widgets.Button(description="Clear Search", button_style='warning')
search_output = widgets.Output()

def execute_search(b):
    with search_output:
        clear_output()
        query = search_input.value.strip()
        if not query:
            print("Error: Please enter a search query.")
            return

        print(f"Searching for information on: '{query}'... (Loading...)") # Added loading feedback
        try:
            rag_output = rag_pipeline_updated(query)
            if isinstance(rag_output, dict):
                context = rag_output.get('context', '')
                if not context:
                    print("Sorry, no relevant information found for this query.")
                else:
                    print(f"\nHere's what I found:\n{context[:500]}...") # Display a snippet
            else: # It's a string (error or status message from rag_pipeline_updated)
                print(rag_output)
        except Exception as e:
            print(f"Error running search engine: {e}")

def clear_search(b):
    search_input.value = ''
    with search_output:
        clear_output()

search_button.on_click(execute_search)
clear_button.on_click(clear_search)

search_engine_screen = widgets.VBox([
    search_label, search_input, widgets.HBox([search_button, clear_button]), search_output
], layout=widgets.Layout(border='2px solid #D3D3D3', padding='10px', margin='5px', background_color='#F5F5F5')) # Added styling

# --- Screen 4: Visual Dashboard ---
dash_label = widgets.HTML(value="<h3>Screen 4: Orchid Health Dashboard (Based on Historical Data)</h3>")
refresh_button = widgets.Button(description="Refresh Graphs", button_style='success')
dash_output = widgets.Output()

def draw_dashboard(b=None):
    with dash_output:
        clear_output(wait=True)
        days = ['Day 1', 'Day 2', 'Day 3', 'Day 4', 'Day 5', 'Day 6', 'Day 7']
        health_index = np.random.randint(60, 100, size=7)
        moisture_history = np.random.randint(30, 70, size=7)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.plot(days, health_index, marker='o', color='green', linestyle='-')
        ax1.set_title('Plant Health Index (%)')
        ax1.set_ylim(0, 100)
        ax1.grid(True, linestyle='--', alpha=0.6)

        ax2.bar(days, moisture_history, color='skyblue')
        ax2.set_title('Soil Moisture History (%)')
        ax2.set_ylim(0, 100)

        plt.tight_layout()
        plt.show()

refresh_button.on_click(draw_dashboard)
visual_dashboard_screen = widgets.VBox([
    dash_label, refresh_button, dash_output
], layout=widgets.Layout(border='2px solid #CCEECC', padding='10px', margin='5px', background_color='#EEFFEE')) # Added styling

# --- Display all screens linearly for tab navigation ---
display(widgets.HTML(value="<h3>Navigating Screens:</h3><p>The screens are displayed one after another. You can scroll through them and use the `Tab` key on your keyboard to move between the various input fields and buttons on each screen.</p>"))
display(image_upload_screen,
        iot_sensor_screen,
        search_engine_screen,
        visual_dashboard_screen)

sample_button.click()

HTML(value='<h3>Navigating Screens:</h3><p>The screens are displayed one after another. You can scroll through…

In [ ]:
# CELL 8: Initialize RAG System
# =============================
"""
🚀 CELL 8: INITIALIZE RAG SYSTEM
This cell creates the RAG system and loads the sample papers.
"""

# Sample data to initialize the system
SAMPLE_PAPERS = [
    {
        "title": "Deceptive pollination in orchids",
        "abstract": "A review of patterns, mechanisms, and evolution in orchid pollination.",
        "authors": "Smith, J.",
        "journal": "Botany Review",
        "year": 2021
    },
    {
        "title": "Orchid mycorrhizal fungi",
        "abstract": "This study investigates the structure, function, and diversity of fungi in orchid roots.",
        "authors": "Lee, A.",
        "journal": "Mycology Today",
        "year": 2022
    }
]

# Configuration
OPENAI_API_KEY = None

# Initialize the RAG system
print("🌊 Initializing Ecological RAG System...")
rag_system = EcologicalRAG(openai_api_key=OPENAI_API_KEY)

# Load sample papers
print("\n📚 Loading sample papers into RAG system...")
rag_system.load_papers(SAMPLE_PAPERS)

# Test the system
print("\n🧪 Testing system with sample query...")
test_result = rag_system.query("What are the mechanisms of pollination?")
print(f"✅ Test successful! Found {test_result['papers_found']} relevant papers")

print("\n🎉 RAG system is ready!")
print("📋 Next: Run Cell 9 for simple interface or Cell 10 to load your own papers")

🌊 Initializing Ecological RAG System...
🌊 Ecological RAG System Initialized

📚 Loading sample papers into RAG system...
📚 Loading 2 papers...
📖 Found 2 papers with abstracts
🔄 Generating embeddings...
💾 Adding to vector store...
✅ Successfully loaded 2 papers!

🧪 Testing system with sample query...
✅ Test successful! Found 2 relevant papers

🎉 RAG system is ready!
📋 Next: Run Cell 9 for simple interface or Cell 10 to load your own papers


In [ ]:
import json

# Create a dummy JSON file to test the 'load_collected_papers' function
sample_data = [
    {
        "title": "Genomic Diversity of Tropical Orchids",
        "abstract": "This study explores the genomic variations across various tropical orchid species to understand climate adaptation.",
        "authors": "Dr. Orchid",
        "journal": "Nature Botany",
        "year": 2023
    }
]

file_path = 'iolr_2022_abstracts_abstracts_only.json'
with open(file_path, 'w') as f:
    json.dump(sample_data, f, indent=4)

print(f"Successfully created {file_path} with sample data.")

Successfully created iolr_2022_abstracts_abstracts_only.json with sample data.


In [ ]:
# CELL 10: Load Your Own Papers
import json
import pandas as pd

file_path = 'iolr_2022_abstracts_abstracts_only.json'
try:
    with open(file_path, 'r') as f:
        your_papers = json.load(f)

    rag_system = EcologicalRAG() # Uses the fixed class
    rag_system.load_papers(your_papers)

    print("\n📊 Paper loaded into RAG system successfully!")
except Exception as e:
    print(f"❌ Error: {e}")

🌊 Ecological RAG System Initialized
📚 Loading 1 papers...
📖 Found 1 papers with abstracts
🔄 Generating embeddings...
💾 Adding to vector store...
✅ Successfully loaded 1 papers!

📊 Paper loaded into RAG system successfully!


In [ ]:
import fitz  # PyMuPDF
import os

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return ""

# Mapping your local PDFs to your metadata
pdf_folder = '/content/sample_data/'
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

print(f"Found {len(pdf_files)} PDF files in sample_data.")

# Enrich 'your_papers' data with full text if available
if 'your_papers' in globals() and your_papers:
    for paper in your_papers:
        # Simple matching logic: check if a PDF filename contains a keyword from the title
        title_keywords = paper['title'].split()[:3]
        for pdf in pdf_files:
            if any(key.lower() in pdf.lower() for key in title_keywords):
                print(f"Reading full text for: {paper['title']}")
                full_text = extract_text_from_pdf(os.path.join(pdf_folder, pdf))
                # Append full text to abstract for indexing
                paper['abstract'] = paper['abstract'] + "\n\nFULL TEXT:\n" + full_text

    # Re-load the enriched papers using the current class method
    rag_system.load_papers(your_papers)
    print("\n✅ RAG System updated with full PDF content!")

Found 5 PDF files in sample_data.
Reading full text for: Genomic Diversity of Tropical Orchids
Reading full text for: Genomic Diversity of Tropical Orchids
Reading full text for: Genomic Diversity of Tropical Orchids
📚 Loading 1 papers...
📖 Found 1 papers with abstracts
🔄 Generating embeddings...
💾 Adding to vector store...
✅ Successfully loaded 1 papers!

✅ RAG System updated with full PDF content!


In [ ]:
# SYNC AND SEARCH: Re-run this to apply all changes
# =================================================

# 1. Re-initialize the system with the latest class definition
rag_system = EcologicalRAG()

# 2. Re-load your papers (including full text from PDFs if you ran those cells)
if 'your_papers' in globals():
    rag_system.load_papers(your_papers)

    # 3. Perform the query
    print("\n--- Query Result ---")
    query = "What is the genomic diversity of orchids?"
    response = rag_pipeline_updated(query)
    print(response)
else:
    print("❌ 'your_papers' not found. Please run the data loading cells first.")

🌊 Ecological RAG System Initialized
📚 Loading 1 papers...
📖 Found 1 papers with abstracts
🔄 Generating embeddings...
💾 Adding to vector store...
✅ Successfully loaded 1 papers!

--- Query Result ---
--- Retrieved Context from your PDFs ---
Source: Genomic Diversity of Tropical Orchids
Retrieved context length: 266769 characters. Ready for LLM generation.


In [ ]:
# Updated RAG Pipeline using the EcologicalRAG instance and LLM
def rag_pipeline_updated(query):
    if 'rag_system' not in globals():
        return "❌ System not found. Please run the Initialization cell (Cell 8) first."
    if 'llm_model' not in globals():
        return "❌ LLM model not initialized. Please run the Gemini API setup cells first."

    try:
        # Query the live system to get context
        result = rag_system.query(query)
        context = result.get('context', '')

        if not context:
            return "🔍 No relevant information found in your PDFs for this query."

        print("--- Retrieved Context from your PDFs ---")
        for meta in result.get('metadata', []):
            print(f"Source: {meta.get('title', 'Unknown Paper')}")

        # Prepare prompt for LLM
        prompt = f"Given the following context, answer the question. If the answer is not in the context, state that you don't have enough information.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"

        # Generate answer using LLM
        llm_response = llm_model.generate_content(prompt)

        # Check if the response contains 'candidates' and 'text'
        if llm_response and llm_response.candidates and llm_response.candidates[0].content.parts:
            answer = llm_response.candidates[0].content.parts[0].text
        else:
            answer = "Sorry, I could not generate an answer based on the provided context."

        return answer

    except Exception as e:
        return f"❌ Error: {str(e)}. Please try re-running the initialization cell and Gemini API setup."

user_query = "What is the genomic diversity of orchids?"
print(rag_pipeline_updated(user_query))

--- Retrieved Context from your PDFs ---
Source: Genomic Diversity of Tropical Orchids
Based on the provided context, the study explores the genomic variations across various tropical orchid species to understand climate adaptation. It notes that researchers obtained 1,450 low-copy nuclear genes from 610 orchid species (including 431 with newly generated transcriptomes) to reconstruct their phylogenetic relationships. 

However, the text does not provide specific details describing the exact nature or characteristics of the genomic diversity of orchids itself. Therefore, I don't have enough information to provide a more detailed answer.
